# Training_Best_AdamW_vs_Prodigy_30ep.ipynb

**Branch**: `Visualizer_Building` (creato post-merge consolidato).

**Contesto**: dopo l'audit delle run pre-EventProp, sono emerse delle scoperte importanti che richiedevano un train piu' approfondito:
- `P15_S2E_A8_attn` (Spike Attention): val_data **0.180** a ep15, trend ancora in discesa
- `P12_S2D_F7` (baseline puro, no OU/SR/Po2): val_data **0.214** a ep15
- `P15_S2E_A1_baseline` (default config): val_data **0.220** a ep15
- `eventprop_alif_full` (5 ep): val_data **0.223**
- AdamW vs Prodigy: pareggio sostanziale entro 0.001

**Obiettivo notebook**: training rigoroso a **30 epoche** delle 4 architetture top con AdamW E Prodigy, naming chiaro, simulator integrato.

## Naming convention nuovo

| Tag | Architettura | Lambdas PINN | Po2 |
|---|---|---|---|
| `BASE_FULL` | A1 baseline ALIF (32 neuroni, rank=8, delays=6) | data + phys + ou + bc | ON |
| `BASE_PURE` | A1 baseline ALIF (= F7: no OU/SR/Po2) | solo data | OFF |
| `ATTN`      | A8 Spike Attention (3936 params) | data + phys + ou + bc | ON |
| `EVENTPROP_ALIF` | A1 architecture + EventProp adjoint | data + phys + ou + bc | ON |

Per ogni architettura testiamo 2 optimizer (8 run totali):
- `_adamw` — AdamW lr=2e-3 + OneCycleLR (matches originali)
- `_prodigy` — Prodigy lr=1.0 d_coef=1.0 + scheduler=none

**Tag completo**: `T30_<ARCH>_<OPT>` — es. `T30_BASE_FULL_adamw`, `T30_ATTN_prodigy`.

## Fix Prodigy lr_eff

Aggiornata `train.py` per loggare per-epoca anche `prodigy_d`, `prodigy_d_max`, `prodigy_lr_eff = lr * d`. Aggiornato `plot_g3_lr_schedule` per mostrare 2-panel se Prodigy detected (LR nominale + LR effettiva + d adapter).

## Configurazione comune

- 30 epoche × 190 step (= 5700 batch totali)
- batch=8, val_batch=64, seq_len=50
- scenario_mix=highway, cut_in=0.0 (matches originali Architecture_Exploration)
- cache: `data/cache_1500_highway_cut0.0_ou0.0.pt` (noise_scale=0.0)
- h=32 r=8 (default)
- po2_enabled=1 per BASE_FULL/ATTN/EVENTPROP_ALIF; 0 per BASE_PURE

**Tempi Azure stimati**: ~6h totali (8 runs × ~45 min avg). Adatto a esecuzione overnight.

**SKIP_IF_EXISTS**: rilancio sicuro dopo interruzioni. Push per-run.

## Output
- `results/T30_<TAG>/` per ogni run (CSV epoca + batch + plot G1-G13 + best_model.pt)
- Cella 8 (analisi): pivot val_data 4×2 + boxplot per metric operative
- Cella 9 (simulator): per ogni best config esegue CFSimulator + plot static + GIF cut-in

In [ ]:
# ===========================================================
# CELLA 0 -- Bootstrap deps + git checkout Visualizer_Building
# ===========================================================
import sys

print('Installing/upgrading dependencies (idempotent)...')
!{sys.executable} -m pip install --quiet matplotlib pandas pillow
!{sys.executable} -m pip install --quiet nbstripout
!{sys.executable} -m pip install --quiet prodigyopt   # per --optimizer prodigy
!nbstripout --install --attributes .gitattributes 2>/dev/null
print('   [OK] deps installed + nbstripout attivato')

print('\ngit fetch + checkout Visualizer_Building + pull:')
!git fetch origin
!git checkout Visualizer_Building 2>/dev/null || git checkout -b Visualizer_Building origin/Visualizer_Building
!git pull --no-rebase --no-edit origin Visualizer_Building

print('\nBranch + commit attuale:')
!git branch --show-current
!git log --oneline -3

In [ ]:
# ===========================================================
# CELLA 1 -- ENV CHECK + verifica train.py CLI + build_model
# ===========================================================
import sys, os, platform, subprocess

print(f'Python:          {sys.version.split()[0]} ({platform.system()})')
print(f'Working dir:     {os.getcwd()}')

checks = []
for mod in ['torch', 'numpy', 'pandas', 'matplotlib', 'prodigyopt']:
    try:
        m = __import__(mod)
        v = getattr(m, '__version__', '?')
        print(f'   [OK] {mod:<14} v{v}')
        checks.append((mod, True))
    except Exception as e:
        print(f'   [FAIL] {mod:<14} {e}')
        checks.append((mod, False))

import torch
print(f'\nCUDA: ' + ('[OK] ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else '[INFO] CPU only'))

print('\nFile critici:')
for f in ['train.py', 'config.py', 'core/network.py', 'core/eventprop.py',
          'utils/plot_diagnostics.py', 'utils/simulator/engine.py']:
    exists = os.path.isfile(f)
    print(f'   {"[OK]" if exists else "[MISSING]"} {f}')
    checks.append((f, exists))

# Verifica CLI flags critici
res = subprocess.run([sys.executable, 'train.py', '--help'], capture_output=True, text=True)
help_text = res.stdout + res.stderr
for flag in ['--training_method', '--prodigy_d_coef', '--noise_scale', '--po2_enabled',
             '--max_steps_per_epoch', '--cf_hidden_size', '--cf_rank']:
    ok = flag in help_text
    print(f'   {"[OK]" if ok else "[MISSING]"} train.py supporta {flag}')
    checks.append((flag, ok))

# Verifica le 4 varianti che useremo
from core.network import build_model
for v in ['baseline', 'attn', 'eventprop_alif_full']:
    try:
        m = build_model(v)
        n = sum(p.numel() for p in m.parameters())
        print(f'   [OK] build_model({v:25s}) params={n:>5,d}')
    except Exception as e:
        print(f'   [FAIL] build_model({v}): {e}')
        checks.append((v, False))

n_fail = sum(1 for _, ok in checks if not ok)
print(f'\n{"[OK]" if n_fail == 0 else f"[FAIL] {n_fail} problemi"} ENV CHECK')
if n_fail > 0:
    raise SystemExit('Env check failed')

In [ ]:
# ===========================================================
# CELLA 2 -- TRAIN_PLAN definition (4 arch x 2 opt = 8 run)
# ===========================================================
# Tutte le configurazioni condividono il setup comune (highway, cache F2,
# 30ep x 190 step, batch=8, h=32 r=8). Cambia solo:
#  - training_method (per architettura)
#  - optimizer + lr + scheduler
#  - po2_enabled e lambdas (BASE_PURE = solo data, others = full PINN)

import time, subprocess, sys, os, datetime, shutil, glob, json
import pandas as pd, numpy as np

COMMON = {
    'epochs':              30,
    'max_steps_per_epoch': 190,
    'batch_size':          8,
    'val_batch_size':      64,
    'seq_len':             50,
    'scenario_mix':        'highway',
    'cut_in_ratio':        0.0,
    'cf_hidden_size':      32,
    'cf_rank':             8,
    'noise_scale':         0.0,        # matches A1/A8/F7 Architecture_Exploration
    'n_train':             1500,
    'n_val':               300,
    'max_inf_streak':      99999,      # no abort
    'early_stop_patience': 0,          # no early stop, run full 30 ep
}
CACHE_PATH = f'data/cache_{COMMON["n_train"]}_{COMMON["scenario_mix"]}_cut{COMMON["cut_in_ratio"]}_ou{COMMON["noise_scale"]}.pt'

# 4 ARCHITETTURE (matches l'analisi pre-EventProp):
ARCHITECTURES = {
    'BASE_FULL': {
        'training_method': 'baseline',
        'po2_enabled':     1,
        'lambda_data':     1.0,
        'lambda_phys':     0.1,
        'lambda_ou':       0.05,
        'lambda_bc':       1.0,
        'lambda_sr':       0.0,
        '_desc':           'A1 baseline ALIF + Po2 + PINN multi-obj (production default)',
    },
    'BASE_PURE': {
        'training_method': 'baseline',
        'po2_enabled':     0,           # matches F7
        'lambda_data':     1.0,
        'lambda_phys':     0.0,
        'lambda_ou':       0.0,
        'lambda_bc':       0.0,
        'lambda_sr':       0.0,
        '_desc':           'F7-style baseline: solo data loss, NO Po2 (test floor pulito)',
    },
    'ATTN': {
        'training_method': 'attn',
        'po2_enabled':     1,
        'lambda_data':     1.0,
        'lambda_phys':     0.1,
        'lambda_ou':       0.05,
        'lambda_bc':       1.0,
        'lambda_sr':       0.0,
        '_desc':           'A8 Spike Attention (3936 params) + PINN multi-obj',
    },
    'EVENTPROP_ALIF': {
        'training_method': 'eventprop_alif_full',
        'po2_enabled':     1,
        'lambda_data':     1.0,
        'lambda_phys':     0.1,
        'lambda_ou':       0.05,
        'lambda_bc':       1.0,
        'lambda_sr':       0.0,
        '_desc':           'A1 architecture + EventProp adjoint (training method nuovo)',
    },
}

# 2 OPTIMIZERS:
OPTIMIZERS = {
    'adamw': {
        'optimizer':       'adamw',
        'lr':              2e-3,
        'scheduler':       'onecycle',
        'max_lr':          2e-3,
        'prodigy_d_coef':  1.0,        # ignored
        '_desc':           'AdamW lr=2e-3 + OneCycleLR (matches Architecture_Exploration originali)',
    },
    'prodigy': {
        'optimizer':       'prodigy',
        'lr':              1.0,        # canonical Prodigy
        'scheduler':       'none',     # Prodigy auto-tune via d adapter
        'max_lr':          1.0,        # unused
        'prodigy_d_coef':  1.0,        # canonical, no brake
        '_desc':           'Prodigy lr=1.0 d_coef=1.0 (canonical) + scheduler=none',
    },
}

# Build full plan: 4 arch x 2 opt = 8 runs
TRAIN_PLAN = []
for arch_name, arch_cfg in ARCHITECTURES.items():
    for opt_name, opt_cfg in OPTIMIZERS.items():
        tag = f'T30_{arch_name}_{opt_name}'
        full_cfg = {**COMMON, **arch_cfg, **opt_cfg, 'tag': tag}
        TRAIN_PLAN.append(full_cfg)

# Filter run list (modifica per disabilitare singoli plan)
RUN_TAGS = set(p['tag'] for p in TRAIN_PLAN)  # default: TUTTI

print(f'TRAIN PLAN: {len(TRAIN_PLAN)} run totali')
print(f'Cache: {CACHE_PATH}  ({"esistente" if os.path.isfile(CACHE_PATH) else "DA GENERARE (rigenerato durante run 1)"})')
print(f'Setup: 30ep x 190step, batch=8, h=32 r=8, scenario=highway, noise=0, cache F2-style')
print()
for p in TRAIN_PLAN:
    in_run = '[RUN]' if p['tag'] in RUN_TAGS else '[SKIP]'
    print(f'  {in_run} {p["tag"]:<35s} -- {p["_desc"]}')
    print(f'           optimizer={p["optimizer"]:7s} lr={p["lr"]:>5} scheduler={p["scheduler"]:>8s} po2={p["po2_enabled"]} lambdas=({p["lambda_data"]},{p["lambda_phys"]},{p["lambda_ou"]},{p["lambda_bc"]})')

In [ ]:
# ===========================================================
# CELLA 3 -- Cache check + auto-generate if missing
# ===========================================================
if not os.path.isfile(CACHE_PATH):
    print(f'[CACHE GEN] {CACHE_PATH} mancante, generazione automatica...')
    from data.generator import generate_dataset
    from config import SEED
    import torch
    t0 = time.time()
    train_data = generate_dataset(COMMON['n_train'], base_seed=SEED,
                                    scenario_mix={'highway': 1.0},
                                    cut_in_ratio=COMMON['cut_in_ratio'],
                                    noise_scale=COMMON['noise_scale'])
    val_data = generate_dataset(COMMON['n_val'], base_seed=SEED + 1,
                                  scenario_mix={'highway': 1.0},
                                  cut_in_ratio=COMMON['cut_in_ratio'],
                                  noise_scale=COMMON['noise_scale'])
    torch.save({'train': train_data, 'val': val_data, 'seed': SEED}, CACHE_PATH)
    print(f'[CACHE GEN] saved {CACHE_PATH} ({os.path.getsize(CACHE_PATH)/1024/1024:.1f} MB) in {time.time()-t0:.0f}s')
else:
    sz = os.path.getsize(CACHE_PATH) / 1024 / 1024
    print(f'[CACHE OK] {CACHE_PATH} esistente ({sz:.1f} MB)')

In [ ]:
# ===========================================================
# CELLA 4 -- RUN sweep loop (SKIP_IF_EXISTS + push per-run)
# ===========================================================
# SKIP_IF_EXISTS=True: salta plan con results/<tag>/training_log.csv esistente
# Permette di rilanciare la cella dopo crash/interruzioni senza ripetere lavoro.
SKIP_IF_EXISTS = True

def _build_cli(p):
    args = [
        sys.executable, 'train.py',
        '--training_method',     p['training_method'],
        '--epochs',              str(p['epochs']),
        '--n_train',             str(p['n_train']),
        '--n_val',               str(p['n_val']),
        '--batch_size',          str(p['batch_size']),
        '--val_batch_size',      str(p['val_batch_size']),
        '--seq_len',             str(p['seq_len']),
        '--scheduler',           p['scheduler'],
        '--max_lr',              str(p['max_lr']),
        '--lr',                  str(p['lr']),
        '--optimizer',           p['optimizer'],
        '--prodigy_d_coef',      str(p['prodigy_d_coef']),
        '--scenario_mix',        p['scenario_mix'],
        '--cut_in_ratio',        str(p['cut_in_ratio']),
        '--cf_hidden_size',      str(p['cf_hidden_size']),
        '--cf_rank',             str(p['cf_rank']),
        '--lambda_data',         str(p['lambda_data']),
        '--lambda_phys',         str(p['lambda_phys']),
        '--lambda_ou',           str(p['lambda_ou']),
        '--lambda_bc',           str(p['lambda_bc']),
        '--lambda_sr',           str(p['lambda_sr']),
        '--noise_scale',         str(p['noise_scale']),
        '--po2_enabled',         str(p['po2_enabled']),
        '--max_inf_streak',      str(p['max_inf_streak']),
        '--early_stop_patience', str(p['early_stop_patience']),
        '--max_steps_per_epoch', str(p['max_steps_per_epoch']),
        '--data_cache',          CACHE_PATH,
        '--tag',                 p['tag'],
    ]
    return args

def _push_run(p):
    tag = p['tag']
    src, dst = f'checkpoints/{tag}', f'results/{tag}'
    if not os.path.isdir(src):
        print(f'   [WARN] {src} mancante, niente da pushare')
        return False
    if os.path.isdir(dst): shutil.rmtree(dst)
    os.makedirs(f'{dst}/plots', exist_ok=True)
    for f in glob.glob(f'{src}/*.csv') + glob.glob(f'{src}/*.json'):
        shutil.copy2(f, dst)
    for f in glob.glob(f'{src}/plots/*.png'):
        shutil.copy2(f, f'{dst}/plots/')

    val_str = ''
    if os.path.isfile(f'{dst}/training_log.csv'):
        edf = pd.read_csv(f'{dst}/training_log.csv')
        if len(edf) > 0:
            best_idx = int(edf.val_data.idxmin())
            val_str = f'best val_data={edf.val_data.min():.4f} (E{best_idx+1}/{len(edf)})'

    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M')
    msg = (
        f'results (T30 Best AdamW vs Prodigy 30ep): {tag} ({ts})\n\n'
        f'{val_str}\n'
        f'method={p["training_method"]} optimizer={p["optimizer"]} '
        f'lr={p["lr"]} sched={p["scheduler"]} po2={p["po2_enabled"]}\n'
        f'Branch Visualizer_Building\n'
    )
    with open('/tmp/t30_msg.txt', 'w', encoding='utf-8') as f:
        f.write(msg)
    !git add {dst}
    !git commit -F /tmp/t30_msg.txt
    !rm /tmp/t30_msg.txt
    !git pull --no-rebase --no-edit origin Visualizer_Building
    !git push origin Visualizer_Building
    return True

# ── Esecuzione ────────────────────────────────────
run_results = []
t_start = time.time()
run_idx = 0
total_runs = sum(1 for p in TRAIN_PLAN if p['tag'] in RUN_TAGS)

for p in TRAIN_PLAN:
    if p['tag'] not in RUN_TAGS:
        continue
    run_idx += 1
    tag = p['tag']

    if SKIP_IF_EXISTS and os.path.isfile(f'results/{tag}/training_log.csv'):
        try:
            ep = pd.read_csv(f'results/{tag}/training_log.csv')
            v_str = f'val_data={ep.val_data.min():.4f}' if len(ep) else 'empty'
        except Exception:
            v_str = 'unreadable'
        print(f'\n[{run_idx}/{total_runs}] [SKIP_EXIST] {tag}: ({v_str}). '
              f'Set SKIP_IF_EXISTS=False per ri-eseguire.')
        run_results.append({'tag': tag, 'status': 'skipped', 'elapsed': 0.0})
        continue

    print('\n' + '=' * 78)
    print(f'[{run_idx}/{total_runs}] {tag}')
    print(f'  {p["_desc"]}')
    print(f'  optimizer={p["optimizer"]} lr={p["lr"]} sched={p["scheduler"]} po2={p["po2_enabled"]}')
    print('=' * 78)
    cmd = _build_cli(p)
    t0 = time.time()
    res = subprocess.run(cmd, capture_output=False)
    elapsed = time.time() - t0
    status = 'ok' if res.returncode == 0 else f'fail({res.returncode})'
    elapsed_total = time.time() - t_start
    eta_min = (elapsed_total / run_idx) * (total_runs - run_idx) / 60
    print(f'\n[{run_idx}/{total_runs}] {tag} -> {status} in {elapsed/60:.1f} min   '
          f'total={elapsed_total/60:.0f}min  ETA={eta_min:.0f}min')
    print(f'\nPush results/{tag}...')
    _push_run(p)
    run_results.append({'tag': tag, 'status': status, 'elapsed': elapsed})

print(f'\n{"="*78}\nSWEEP TRAINING COMPLETO in {(time.time()-t_start)/60:.0f} min\n{"="*78}')
for r in run_results:
    if r['status'] != 'skipped':
        print(f"   {r['tag']:<35} {r['status']:<15} {r['elapsed']/60:.1f}min")

In [ ]:
# ===========================================================
# CELLA 5 -- ANALISI: pivot 4x2 val_data + trajectory overlay
# ===========================================================
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

rows = []
for p in TRAIN_PLAN:
    arch = p['tag'].replace('T30_', '').rsplit('_', 1)[0]
    opt = p['tag'].rsplit('_', 1)[1]
    log = f'results/{p["tag"]}/training_log.csv'
    if not os.path.isfile(log):
        rows.append({'tag': p['tag'], 'arch': arch, 'opt': opt,
                     'val_data_best': None, 'best_ep': None, 'n_ep': 0,
                     'spike_rate_last': None})
        continue
    ep = pd.read_csv(log)
    if len(ep) == 0: continue
    best_ep = int(ep.val_data.idxmin()) + 1
    rows.append({
        'tag': p['tag'], 'arch': arch, 'opt': opt,
        'val_data_best': float(ep.val_data.min()),
        'val_data_last': float(ep.val_data.iloc[-1]),
        'best_ep': best_ep, 'n_ep': len(ep),
        'spike_rate_last': float(ep.spike_rate.iloc[-1]),
        'prodigy_lr_eff_last': float(ep.get('prodigy_lr_eff', pd.Series([float('nan')])).iloc[-1]) if 'prodigy_lr_eff' in ep.columns else None,
    })
df_runs = pd.DataFrame(rows)

if df_runs.val_data_best.notna().any():
    # Pivot 4 arch x 2 opt
    pivot = df_runs.pivot(index='arch', columns='opt', values='val_data_best')
    ARCH_ORDER = ['BASE_FULL', 'BASE_PURE', 'ATTN', 'EVENTPROP_ALIF']
    OPT_ORDER  = ['adamw', 'prodigy']
    pivot = pivot.reindex(ARCH_ORDER)[OPT_ORDER]

    display(Markdown('## TABELLA val_data BEST (RMSE m/s²) -- 4 arch × 2 opt'))
    display(pivot.round(4))

    # Pivot best_ep
    pivot_ep = df_runs.pivot(index='arch', columns='opt', values='best_ep').reindex(ARCH_ORDER)[OPT_ORDER]
    display(Markdown('## Best epoch (= quando il modello ha raggiunto val_data minima)'))
    display(pivot_ep)

    # Pivot val_data_last (ep30) per vedere se ancora in discesa
    pivot_last = df_runs.pivot(index='arch', columns='opt', values='val_data_last').reindex(ARCH_ORDER)[OPT_ORDER]
    display(Markdown('## val_data a ep30 (last) -- se ≈ best, plateau; se ≫ best, sta divergendo'))
    display(pivot_last.round(4))

    # Best assoluto + verdetto AdamW vs Prodigy
    display(Markdown('## Verdetto'))
    best_row = df_runs.loc[df_runs.val_data_best.idxmin()]
    print(f'BEST ASSOLUTO: {best_row.tag}  val_data={best_row.val_data_best:.4f} a ep{best_row.best_ep}')
    for arch in ARCH_ORDER:
        sub = df_runs[df_runs.arch == arch]
        if len(sub) < 2 or sub.val_data_best.isna().any(): continue
        adamw_v = sub[sub.opt == 'adamw'].val_data_best.iloc[0]
        prodigy_v = sub[sub.opt == 'prodigy'].val_data_best.iloc[0]
        delta = prodigy_v - adamw_v
        winner = 'AdamW' if delta > 0 else 'Prodigy'
        print(f'  {arch:<18}: AdamW={adamw_v:.4f}  Prodigy={prodigy_v:.4f}  delta={delta:+.4f}  -> {winner} vince')
else:
    print('Nessun risultato disponibile -- esegui Cella 4 prima.')

In [ ]:
# ===========================================================
# CELLA 6 -- Trajectory overlay: val_data per epoca (8 curve sovrapposte)
# ===========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, opt_name in zip(axes, ['adamw', 'prodigy']):
    for arch_name in ARCH_ORDER:
        tag = f'T30_{arch_name}_{opt_name}'
        log = f'results/{tag}/training_log.csv'
        if not os.path.isfile(log): continue
        ep = pd.read_csv(log)
        if len(ep) == 0: continue
        ax.plot(ep.epoch, ep.val_data, 'o-', linewidth=1.7, markersize=5, label=arch_name)
    ax.set_xlabel('epoch'); ax.set_ylabel('val_data (RMSE m/s²)')
    ax.set_title(f'Optimizer: {opt_name.upper()}')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=10)
fig.suptitle('val_data per epoca (30 ep, 4 architetture)', fontsize=12, fontweight='bold')
plt.tight_layout()
out_traj = f'opt_plots/t30_trajectory_overlay.png'
os.makedirs('opt_plots', exist_ok=True)
fig.savefig(out_traj, dpi=120, bbox_inches='tight')
plt.show()
print(f'[saved] {out_traj}')

In [ ]:
# ===========================================================
# CELLA 7 -- Simulator: per ogni best config, run + plot static
# ===========================================================
# Per ogni run completato, carica best_model.pt + esegue CFSimulator su
# scenari val cut-in (cache mixed) per validazione operativa.
from utils.simulator import CFSimulator, compute_operational_metrics, plot_simulation_static, animate_scenario, save_animation

# Cache mixed con cut-in (gia' generata, vedi simulator notebook)
SIM_CACHE = 'data/cache_400_mixed_cut0.3_ou0.0.pt'
if not os.path.isfile(SIM_CACHE):
    print(f'[GEN] {SIM_CACHE} mancante, genero ...')
    from data.generator import generate_dataset
    from config import SEED
    import torch
    train_d = generate_dataset(300, base_seed=SEED, scenario_mix=None,
                                cut_in_ratio=0.30, noise_scale=0.0)
    val_d = generate_dataset(100, base_seed=SEED+1, scenario_mix=None,
                              cut_in_ratio=0.30, noise_scale=0.0)
    torch.save({'train': train_d, 'val': val_d, 'seed': SEED}, SIM_CACHE)
    print(f'[saved] {SIM_CACHE}')

os.makedirs('opt_plots/t30_simviz', exist_ok=True)

# Map tag arch -> CFSimulator variant
_arch_to_variant = {
    'BASE_FULL': 'baseline',
    'BASE_PURE': 'baseline',
    'ATTN':      'attn',
    'EVENTPROP_ALIF': 'eventprop_alif_full',
}

sim_summaries = []
for p in TRAIN_PLAN:
    tag = p['tag']
    arch = tag.replace('T30_', '').rsplit('_', 1)[0]
    opt  = tag.rsplit('_', 1)[1]
    ckpt = f'checkpoints/{tag}/best_model.pt'
    if not os.path.isfile(ckpt):
        # provo da results/ se presente best_model.pt (di solito gitignored)
        ckpt = f'results/{tag}/best_model.pt'
    if not os.path.isfile(ckpt):
        print(f'[SKIP] {tag}: best_model.pt non trovato')
        continue
    try:
        sim = CFSimulator(
            checkpoint_path=ckpt,
            cache_path=SIM_CACHE,
            variant=_arch_to_variant[arch],
            seq_len=200,  # 20s window
        )
    except Exception as e:
        print(f'[ERR] CFSimulator init failed per {tag}: {e}')
        continue
    df_sc = sim.list_scenarios()
    # Pick: 1 highway no-cut-in + 1 cut-in (se disponibile)
    no_cutin_idx = df_sc[~df_sc.is_cut_in].index[0] if (~df_sc.is_cut_in).any() else None
    cutin_idx = df_sc[df_sc.is_cut_in].index[0] if df_sc.is_cut_in.any() else None
    for label, idx in [('normal', no_cutin_idx), ('cutin', cutin_idx)]:
        if idx is None: continue
        r = sim.simulate_scenario(int(idx))
        m = compute_operational_metrics(r)
        out_png = f'opt_plots/t30_simviz/{tag}_{label}_idx{idx:03d}.png'
        fig = plot_simulation_static(r, metrics=m)
        fig.savefig(out_png, dpi=100, bbox_inches='tight')
        plt.close(fig)
        sim_summaries.append({
            'tag': tag, 'label': label, 'idx': int(idx),
            'val_data_train': df_runs[df_runs.tag == tag].val_data_best.iloc[0] if not df_runs[df_runs.tag == tag].empty else None,
            'gap_rmse': m['gap_rmse_m'], 'pos_drift': m['pos_cum_err_m'],
            'accel_rmse': m['accel_rmse_masked'], 'jerk_max': m['jerk_max_pred'],
            'spike_avg': m['spike_rate_avg'],
        })
        print(f'  [{tag} {label}] idx={idx} gap_rmse={m["gap_rmse_m"]:.2f} pos_drift={m["pos_cum_err_m"]:.2f} '
              f'accel_rmse={m["accel_rmse_masked"]:.3f} sr={m["spike_rate_avg"]*100:.1f}% [{out_png}]')
    del sim  # free model

df_sim = pd.DataFrame(sim_summaries)
if len(df_sim) > 0:
    display(Markdown('## Simulator summary (per tag × scenario)'))
    display(df_sim.round(3))
    df_sim.to_csv('opt_plots/t30_simviz/summary.csv', index=False)
    print(f'\n[saved] opt_plots/t30_simviz/summary.csv')